In [1]:
import os
import pandas as pd
import numpy as np
from joblib import load
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from src.features import build_features
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

c:\Users\khandelw2\OneDrive - Coventry University\Data-Driven-Based-Simulation\REFIT-Data\src\features\data_utils.py:6: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


In [2]:
building_id = "Building01"
data = pd.read_pickle(f"data/processed/{building_id}_Processed.pkl")

In [3]:
print(data.shape)

(24606, 54)


In [4]:
data.head()

,Space13_Relative_humidity,Space13_Surface_temperature,Space13_Air_temperature,Space17_Surface_temperature,Space17_Air_temperature,Space14_Air_temperature,Space14_Surface_temperature,Space10_Air_temperature,Space4_Air_temperature,Space6_Relative_humidity,...,Space9_Air_temperature_1,Space9_Air_temperature_2,Space9_Relative_humidity_1,Space9_Relative_humidity_2,Space1_Air_temperature_1,Space1_Air_temperature_2,Space15_Relative_humidity_1,Space15_Relative_humidity_2,Space15_Air_temperature_1,Space15_Air_temperature_2
Date,,,,,,,,,,,,,,,,,,,,,
2013-11-04 10:00:00+00:00,67.241,34.110,16.368,25.139,20.2340,18.6635,33.563,18.6160,22.9575,47.175,...,19.459999,19.912001,53.285000,52.509998,19.044001,18.996500,57.553001,58.415001,19.792999,20.031000
2013-11-04 10:30:00+00:00,67.448,30.115,16.225,23.138,20.3290,18.4735,27.570,18.5685,20.7100,47.160,...,19.174999,19.459999,54.140999,53.285000,19.234501,19.044001,57.394001,57.553001,19.555000,19.792999
2013-11-04 11:00:00+00:00,67.702,27.617,16.177,22.638,20.4715,18.3785,24.570,18.6635,23.0520,47.788,...,19.127001,19.174999,54.257999,54.140999,19.566999,19.234501,57.166000,57.394001,19.459999,19.555000
2013-11-04 11:30:00+00:00,67.839,25.617,16.129,22.638,20.5670,18.3310,23.070,18.9010,20.9485,48.095,...,19.127001,19.127001,54.134998,54.257999,19.900499,19.566999,57.032001,57.166000,19.389000,19.459999
2013-11-04 12:00:00+00:00,67.960,24.617,16.153,22.138,20.6150,18.3310,22.069,19.1395,23.1020,48.567,...,19.198999,19.127001,54.021000,54.134998,20.186001,19.900499,57.120998,57.032001,19.365000,19.389000


In [5]:
data.drop(columns=["Space8_Air_temperature", "Space8_Air_temperature_1", "Space8_Air_temperature_2"], inplace=True)

In [6]:
features = [
    "Air_temperature",
    "Relative_humidity",
    "Surface_temperature",
    "Gas_volume",
    "Outside_air_temperature",
    "Outside_relative_humidity",
    "Wind_speed",
    "Total_rainfall",
]

filtered_columns = [col for col in data.columns if any(col.endswith(feature) for feature in features)]

In [7]:
filtered_columns

['Space13_Relative_humidity',
 'Space13_Surface_temperature',
 'Space13_Air_temperature',
 'Space17_Surface_temperature',
 'Space17_Air_temperature',
 'Space14_Air_temperature',
 'Space14_Surface_temperature',
 'Space10_Air_temperature',
 'Space4_Air_temperature',
 'Space6_Relative_humidity',
 'Space6_Surface_temperature',
 'Space6_Air_temperature',
 'Space8_Air_temperature',
 'Space9_Surface_temperature',
 'Space9_Air_temperature',
 'Space9_Relative_humidity',
 'Space1_Air_temperature',
 'Space15_Relative_humidity',
 'Space15_Air_temperature',
 'Space15_Surface_temperature',
 'Space16_Surface_temperature',
 'Gas_volume',
 'Outside_air_temperature',
 'Outside_relative_humidity',
 'Wind_speed',
 'Total_rainfall']

In [8]:
target_columns = [
        col
        for col in data.columns
        if any(col.endswith(word) for word in ["Air_temperature", "Relative_humidity"])
    ]

In [9]:
X, y = data.drop(columns=target_columns), data[target_columns]

In [10]:
def standard_scale(train, test=None):
    """
    Scale features using standard scaler.

    :param X_train: Training input features.
    :param X_test: Testing input features.
    :return: Scaled training and testing features.
    """
    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(train)
    if test is not None:
        test_scaled = scaler.transform(test)
        return train_scaled, test_scaled, scaler
    
    return train_scaled, scaler

In [11]:
unique_months = data.index.to_period('M').unique()
print(unique_months)

PeriodIndex(['2013-11', '2013-12', '2014-01', '2014-02', '2014-03', '2014-04',
             '2014-05', '2014-06', '2014-07', '2014-08', '2014-09', '2014-10',
             '2014-11', '2014-12', '2015-01', '2015-02', '2015-03', '2015-04',
             '2015-05'],
            dtype='period[M]', name='Date')


In [12]:
test_start_date = '2014-12-01'
test_end_date = '2015-03-31'

X_train = X[(X.index < test_start_date) | (X.index > test_end_date)]
y_train = y[(y.index < test_start_date) | (y.index > test_end_date)]
X_test = X[(X.index >= test_start_date) & (X.index <= test_end_date)]
y_test = y[(y.index >= test_start_date) & (y.index <= test_end_date)]

print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

(18899, 40) (18899, 14) (5707, 40) (5707, 14)


In [13]:
LR_model = LinearRegression()
LR_model.fit(X_train, y_train)

y_pred = LR_model.predict(X_test)


mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2_s = r2_score(y_test, y_pred)

print(mae, mse, rmse, r2_s)

0.21881250105161598 0.2648543047302184 0.5146399758376903 0.984812511380853


In [14]:
X_train_scale, X_test_scale, scaler_X = standard_scale(X_train, X_test)
y_train_scale, y_test_scale, scaler_y = standard_scale(y_train, y_test)

LR_model = LinearRegression()
LR_model.fit(X_train_scale, y_train_scale)

y_pred_scale = LR_model.predict(X_test_scale)

y_pred = pd.DataFrame(
        scaler_y.inverse_transform(y_pred_scale),
        index=y_test.index,
        columns=y.columns,
    )

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2_s = r2_score(y_test, y_pred)

print("One step Prediction:")
print("MAE: ", mae)
print("MSE: ", mse)
print("RMSE: ", rmse)
print("R2 Score: ", r2_s)
print("*********************\n")

One step Prediction:
MAE:  0.2188125010516171
MSE:  0.26485430473021787
RMSE:  0.5146399758376897
R2 Score:  0.9848125113808529
*********************



In [15]:
temp_cols = [col for col in y_test.columns if col.endswith('_Air_temperature')]
humi_cols = [col for col in y_test.columns if col.endswith('_Relative_humidity')]

y_temp_min = y_test[temp_cols].min().min()
y_temp_max = y_test[temp_cols].max().max()

y_humi_min = y_test[humi_cols].min().min()
y_humi_max = y_test[humi_cols].max().max()

print("Temperature Range:", y_temp_min, " ", y_temp_max)
print("Humidity Range:", y_humi_min, " ", y_humi_max)

Temperature Range: 4.311   26.2435
Humidity Range: 31.071   82.013


In [16]:
n_steps = len(y_test)

for col in y.columns:
    fig, ax = plt.subplots(figsize=(20, 10))
    ax.plot(y_test.index[:n_steps], y_test.iloc[:n_steps][col], label="Actual", color="tab:blue")
    ax.plot(
        y_test.index[:n_steps],
        y_pred.iloc[:n_steps][col],
        label="Predicted",
        linestyle="--",
        color="tab:orange",
    )
    ax.set_title(f"Linear Regression Predictions for {col}")
    ax.legend()
    # if "Air_temperature" in col:
    #     ax.set_ylim([y_temp_min, y_temp_max])
    # if "Relative_humidity" in col:
    #     ax.set_ylim([y_humi_min, y_humi_max])
    dir = "reports/figures/Building01_Pred/one-step"
    os.makedirs(dir, exist_ok=True)
    plt.savefig(f"{dir}/{col}lr_pred.png", dpi=300)
    plt.close()

In [17]:
lag1_features = [col for col in X_test.columns if col.endswith("_1")]
lag2_features = [col for col in X_test.columns if col.endswith("_2")]

print(lag1_features)
print(lag2_features)
print((y.columns).tolist())

['Space13_Relative_humidity_1', 'Space13_Air_temperature_1', 'Space17_Air_temperature_1', 'Space14_Air_temperature_1', 'Space10_Air_temperature_1', 'Space4_Air_temperature_1', 'Space6_Relative_humidity_1', 'Space6_Air_temperature_1', 'Space8_Air_temperature_1', 'Space9_Air_temperature_1', 'Space9_Relative_humidity_1', 'Space1_Air_temperature_1', 'Space15_Relative_humidity_1', 'Space15_Air_temperature_1']
['Space13_Relative_humidity_2', 'Space13_Air_temperature_2', 'Space17_Air_temperature_2', 'Space14_Air_temperature_2', 'Space10_Air_temperature_2', 'Space4_Air_temperature_2', 'Space6_Relative_humidity_2', 'Space6_Air_temperature_2', 'Space8_Air_temperature_2', 'Space9_Air_temperature_2', 'Space9_Relative_humidity_2', 'Space1_Air_temperature_2', 'Space15_Relative_humidity_2', 'Space15_Air_temperature_2']
['Space13_Relative_humidity', 'Space13_Air_temperature', 'Space17_Air_temperature', 'Space14_Air_temperature', 'Space10_Air_temperature', 'Space4_Air_temperature', 'Space6_Relative_hum

In [18]:
def make_multi_step_prediction(model, X_test, scaler_X, scaler_y, y_test_columns, n_steps=None, lags=2):
    predictions_df = pd.DataFrame(columns=y_test_columns, index=X_test.index[:n_steps])
    lagged_features = {"lags_1": [col for col in X_test.columns if col.endswith("_1")],
                    "lags_2": [col for col in X_test.columns if col.endswith("_2")]}
    
    for step in range(n_steps):
        # print("****************", step)
        for l in range(1, lags + 1):
            if step - l >= 0:
                # print("Pred", predictions_df.loc[X_test.index[step - l], :])
                X_test.loc[X_test.index[step], lagged_features[f"lags_{l}"]] = predictions_df.loc[X_test.index[step - l], :].values
                
        # print("Actual", X_test.loc[X_test.index[step]])       
        x_scale = scaler_X.transform(X_test.loc[X_test.index[step], :].values.reshape(1, -1))
        predictions_df.loc[X_test.index[step]] = scaler_y.inverse_transform(model.predict(x_scale))
        
    return predictions_df

In [19]:
def make_multi_step_prediction_horizon(model, X_test, scaler_X, scaler_y, y_test_columns, n_steps=None, lags=2):
    predictions_df = pd.DataFrame(columns=y_test_columns, index=X_test.index)
    lagged_features = {"lags_1": [col for col in X_test.columns if col.endswith("_1")],
                    "lags_2": [col for col in X_test.columns if col.endswith("_2")]}
    
    if n_steps is None:
        n_steps = len(X_test)
    
    for i in range(0, len(X_test), n_steps):
        # print("#####################", i)
        actual_steps = min(n_steps, len(X_test) - i)
        for step in range(actual_steps):
            # print("****************", step)
            for l in range(1, lags + 1):
                if step - l >= 0:
                    X_test.loc[X_test.index[i+step], lagged_features[f"lags_{l}"]] = predictions_df.loc[X_test.index[i+step - l], :].values
                    
            x_scale = scaler_X.transform(X_test.loc[X_test.index[i+step], :].values.reshape(1, -1))
            predictions_df.loc[X_test.index[i+step]] = scaler_y.inverse_transform(model.predict(x_scale))
            
    return predictions_df

In [20]:
horizons = [2+1, 6+1, 12+1, 18+1, 24+1, 48+1, 96+1, len(y_test)]

for h in horizons:
    y_multi_step = make_multi_step_prediction_horizon(LR_model, X_test.copy(), scaler_X, scaler_y, y_test_columns=y.columns, n_steps=h)

    mae = mean_absolute_error(y_test, y_multi_step)
    mse = mean_squared_error(y_test, y_multi_step)
    rmse = np.sqrt(mean_squared_error(y_test, y_multi_step))
    r2_s = r2_score(y_test, y_multi_step)

    print(f"{int(h/2)} Hour Horizon Prediction:")
    print("MAE: ", mae)
    print("MSE: ", mse)
    print("RMSE: ", rmse)
    print("R2 Score: ", r2_s)
    print("*********************\n")

1 Hour Horizon Prediction:
MAE:  0.3458070903780023
MSE:  0.5133003586960913
RMSE:  0.7164498298527897
R2 Score:  0.9686320460903078
*********************

3 Hour Horizon Prediction:
MAE:  0.5374516303929866
MSE:  1.029471681125822
RMSE:  1.0146288390962588
R2 Score:  0.9300010242042627
*********************

6 Hour Horizon Prediction:
MAE:  0.7138522341196495
MSE:  1.5729996247333342
RMSE:  1.2541928180042072
R2 Score:  0.8911031207273649
*********************

9 Hour Horizon Prediction:
MAE:  0.8449765747362136
MSE:  2.0759420823670403
RMSE:  1.4408129935446308
R2 Score:  0.8588402692520745
*********************

12 Hour Horizon Prediction:
MAE:  0.9284860110998062
MSE:  2.3388119492703145
RMSE:  1.5293174782465264
R2 Score:  0.8358051948320601
*********************

24 Hour Horizon Prediction:
MAE:  1.1481659276416933
MSE:  3.1821867594960023
RMSE:  1.7838684815579882
R2 Score:  0.7645092137788269
*********************

48 Hour Horizon Prediction:
MAE:  1.3997308113993374
MSE:  4.40

In [21]:
for n_steps in horizons:
    for col in y.columns:
        fig, ax = plt.subplots(figsize=(20, 10))
        ax.plot(y_test.index[:n_steps], y_test.iloc[:n_steps][col], label="Actual", color="tab:blue")
        ax.plot(
            y_test.index[:n_steps],
            y_multi_step.iloc[:n_steps][col],
            label="Predicted",
            linestyle="--",
            color="tab:orange",
        )
        ax.set_title(f"Linear Regression Predictions for {col}")
        ax.legend()
        # if "Air_temperature" in col:
        #     ax.set_ylim([y_temp_min, y_temp_max])
        # if "Relative_humidity" in col:
        #     ax.set_ylim([y_humi_min, y_humi_max])
        dir = f"reports/figures/Building01_Pred/{int(n_steps/2)} hour-horizon"
        os.makedirs(dir, exist_ok=True)
        plt.savefig(f"{dir}/{col}lr_pred.png", dpi=300)
        plt.close()